# 07 — Preprocessing Pipeline
**ChurnOps Phase 8** — Scikit-Learn ColumnTransformer & Unified Pipeline Architecture  
*Constructing a modular, leak-free preprocessing pipeline fitted exclusively on the training split and reusable at production inference.*

---

### Phase 8 Objectives
1. **Pipeline Architecture:** Unify `DomainFeatureEngineer`, `ColumnTransformer`, imputers, scalers, and one-hot encoders into a single Scikit-Learn `Pipeline`.
2. **Feature Selection Integration:** Automatically drop uninformative/redundant features (`gender`, `PhoneService`) identified in Phase 6.
3. **Scaling & Imputation:** Impute numerics (median) and scale with `StandardScaler`; impute categoricals (mode) and encode with `OneHotEncoder(handle_unknown="ignore")`.
4. **Strict Leakage Prevention (Rule R3):** Fit pipeline parameters solely on `data/processed/train.csv`. The test set remains completely locked.
5. **Inference Readiness:** Verify that a raw customer dictionary/DataFrame seamlessly transforms through the pipeline into the exact 51 model features.

In [1]:
import sys, os
from pathlib import Path

PROJECT_ROOT = Path(os.getcwd()).parent if Path(os.getcwd()).name == 'notebooks' else Path(os.getcwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils.config import get_path, PARAMS
from src.data.preprocessing import (
    build_full_pipeline,
    build_column_transformer,
    fit_and_audit_preprocessing
)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

print("Project Root:", PROJECT_ROOT)

Project Root: C:\Users\Abhishek\OneDrive\Pictures\Desktop\ChurnOps


In [2]:
train_path = PROJECT_ROOT / 'data' / 'processed' / 'train.csv'
train_df = pd.read_csv(train_path)

X_train = train_df.drop(columns=['Churn'])
y_train = train_df['Churn']

print(f"Training Input Data: {X_train.shape[0]:,} rows x {X_train.shape[1]:,} features")
X_train.head(3)

Training Input Data: 5,634 rows x 19 features


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,Male,0,No,No,35,No,No phone service,DSL,No,No,Yes,No,Yes,Yes,Month-to-month,No,Electronic check,49.20,1701.65
1,Male,0,Yes,Yes,15,Yes,No,Fiber optic,Yes,No,No,No,No,No,Month-to-month,No,Mailed check,75.10,1151.55
2,Male,0,Yes,Yes,13,No,No phone service,DSL,Yes,Yes,No,Yes,No,No,Two year,No,Mailed check,40.55,590.35


## 1. Assemble and Fit Preprocessing Pipeline

The pipeline consists of:
1. `DomainFeatureEngineer`: row-wise domain transformations (binned tenure cohorts, total services, payment flags).
2. `ColumnTransformer`:
   - **Drop:** `['gender', 'PhoneService']`
   - **Numeric:** `SimpleImputer(median)` + `StandardScaler()` on `['tenure', 'MonthlyCharges', 'TotalCharges', 'total_services']`
   - **Categorical:** `SimpleImputer(most_frequent)` + `OneHotEncoder(handle_unknown='ignore')` on 14 categoricals
   - **Passthrough:** binary indicators (`SeniorCitizen`, `has_family`, `is_month_to_month`, etc.)

In [3]:
pipeline = build_full_pipeline(scale_numeric=True)
pipeline.fit(X_train, y_train)

X_train_proc = pipeline.transform(X_train)
print(f"Transformed Training Shape: {X_train_proc.shape[0]:,} rows x {X_train_proc.shape[1]:,} features")

col_trans = pipeline.named_steps['preprocessor']
feature_names = col_trans.get_feature_names_out()
print(f"Total Output Features: {len(feature_names)}")

Transformed Training Shape: 5,634 rows x 51 features
Total Output Features: 51


## 2. Inspect Transformed Feature Names & Groupings

In [4]:
feature_groups = {
    'Numeric (Scaled)': [f for f in feature_names if f.startswith('num__')],
    'Categorical (One-Hot)': [f for f in feature_names if f.startswith('cat__')],
    'Binary (Passthrough)': [f for f in feature_names if f.startswith('pass__')]
}

for group_name, feats in feature_groups.items():
    print(f"\n{group_name} ({len(feats)} features):")
    print(feats[:8], "..." if len(feats) > 8 else "")


Numeric (Scaled) (4 features):
['num__tenure', 'num__MonthlyCharges', 'num__TotalCharges', 'num__total_services'] 

Categorical (One-Hot) (41 features):
['cat__Partner_No', 'cat__Partner_Yes', 'cat__Dependents_No', 'cat__Dependents_Yes', 'cat__MultipleLines_No', 'cat__MultipleLines_No phone service', 'cat__MultipleLines_Yes', 'cat__InternetService_DSL'] ...

Binary (Passthrough) (6 features):
['pass__SeniorCitizen', 'pass__is_auto_pay', 'pass__has_fiber_optic', 'pass__has_security_or_techsupport', 'pass__has_family', 'pass__is_month_to_month'] 


## 3. Fitted Scaler Parameters (Learned Strictly on Training Set)

In [5]:
scaler = col_trans.named_transformers_['num'].named_steps['scaler']
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'total_services']

scaler_df = pd.DataFrame({
    'Feature': num_cols,
    'Mean': np.round(scaler.mean_, 2),
    'Std': np.round(np.sqrt(scaler.var_), 2)
})
scaler_df

,Feature,Mean,Std
0,tenure,32.49,24.57
1,MonthlyCharges,64.93,30.14
2,TotalCharges,2299.33,2279.00
3,total_services,3.38,2.07


## 4. Production Inference Simulation

We simulate real-time inference by passing a single raw customer record through the fitted pipeline to confirm end-to-end functionality without dependency on external preprocessing scripts.

In [6]:
# Sample incoming raw customer payload (19 features as received from client)
sample_customer = X_train.iloc[[0]].copy()
print("Incoming raw customer features:")
print(sample_customer[['Contract', 'InternetService', 'MonthlyCharges', 'tenure', 'PaymentMethod']])

# Transform single record
transformed_vector = pipeline.transform(sample_customer)
print(f"\nTransformed feature vector shape: {transformed_vector.shape}")
assert transformed_vector.shape == (1, 51), "Inference vector dimension mismatch!"
print("Inference verification successful: 100% compatible with ML estimators.")

Incoming raw customer features:
         Contract InternetService  MonthlyCharges  tenure     PaymentMethod
0  Month-to-month             DSL            49.2      35  Electronic check

Transformed feature vector shape: (1, 51)
Inference verification successful: 100% compatible with ML estimators.


---

## 5. Phase 8 Summary & Hand-off to Phase 9
- **Pipeline Architecture:** `DomainFeatureEngineer` -> `ColumnTransformer` (Drop, Num, Cat, Pass).
- **Output Dimensionality:** 51 fully numeric features ready for linear and tree estimators.
- **Fitted Artifact:** Serialized to `models/preprocessor.joblib`.
- **Zero Leakage:** All parameters derived exclusively from `data/processed/train.csv`.
- **Ready for:** Phase 9 — Baseline Models (Logistic Regression, Random Forest, XGBoost, and DummyClassifier evaluated via 5-fold CV).